In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import csv

In [2]:
import psutil
print(f"RAM dispo: {psutil.virtual_memory().available / 1024**3:.1f} Go")   

RAM dispo: 9.9 Go


In [3]:
def extraction_data_idf_indus(Fichier1, Fichier2):
    cols_etab = ['siren', 'siret', 'dateCreationEtablissement',
                 'trancheEffectifsEtablissement', 'denominationUsuelleEtablissement', 'dateDernierTraitementEtablissement',
                 'etablissementSiege', 'numeroVoieEtablissement', 'typeVoieEtablissement',
                 'libelleVoieEtablissement', 'codePostalEtablissement',
                 'libelleCommuneEtablissement', 'coordonneeLambertAbscisseEtablissement',
                 'coordonneeLambertOrdonneeEtablissement', 'etatAdministratifEtablissement',
                 'activitePrincipaleEtablissement', 'caractereEmployeurEtablissement', 'codePaysEtrangerEtablissement']

    dep = ('75', '77', '78', '91', '92', '93', '94', '95')
    prefixes_activite = ('10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
                          '26', '27', '28', '29', '30', '31', '32', '33', '35', '36', '37', '38', '39', '41', '42', '43',
                            '69', '70', '71', '72', '73', '74', '75')   # SECTION C, D, E, F, M

    pf = pq.ParquetFile(Fichier1)
    n_batches = (pf.metadata.num_rows + 49_999) // 50_000
    batches = []
    for batch in tqdm(pf.iter_batches(batch_size=50_000, columns=cols_etab),
                      total=n_batches, desc="Établissements"):
        df_batch = batch.to_pandas()
        df_batch = df_batch[
        df_batch['codePostalEtablissement'].str.startswith(dep, na=False)
        &
        # df_batch['activitePrincipaleEtablissement'].str.startswith(prefixes_activite, na=False)
        # &
        df_batch['etatAdministratifEtablissement'].str.startswith("A", na=False)
]
        batches.append(df_batch)
    df_etab = pd.concat(batches, ignore_index=True)

    cols_unit = ['siren', 'denominationUniteLegale', 'sexeUniteLegale', 'prenomUsuelUniteLegale',
                 'trancheEffectifsUniteLegale', 'categorieEntreprise', 'categorieJuridiqueUniteLegale']

    pf_2 = pq.ParquetFile(Fichier2)
    n_batches_2 = (pf_2.metadata.num_rows + 49_999) // 50_000
    batches = []
    for batch in tqdm(pf_2.iter_batches(batch_size=50_000, columns=cols_unit),
                      total=n_batches_2, desc="Unités légales"):
        batches.append(batch.to_pandas())
    df_unit = pd.concat(batches, ignore_index=True)

    resultat = pd.merge(df_etab, df_unit, on='siren', how='left')
    return resultat

In [4]:
df_fusion = extraction_data_idf_indus('data/stock-stocketablissement-parquet.parquet', 'data/stock-stockunitelegale-parquet.parquet')

Unités légales: 100%|██████████| 601/601 [00:20<00:00, 29.85it/s]


In [5]:
df_fusion.info()

<class 'pandas.DataFrame'>
RangeIndex: 2966640 entries, 0 to 2966639
Data columns (total 24 columns):
 #   Column                                  Dtype         
---  ------                                  -----         
 0   siren                                   str           
 1   siret                                   str           
 2   dateCreationEtablissement               object        
 3   trancheEffectifsEtablissement           str           
 4   denominationUsuelleEtablissement        str           
 5   dateDernierTraitementEtablissement      datetime64[us]
 6   etablissementSiege                      bool          
 7   numeroVoieEtablissement                 str           
 8   typeVoieEtablissement                   str           
 9   libelleVoieEtablissement                str           
 10  codePostalEtablissement                 str           
 11  libelleCommuneEtablissement             str           
 12  coordonneeLambertAbscisseEtablissement  str          

In [6]:
df_fusion.head()

,siren,siret,dateCreationEtablissement,trancheEffectifsEtablissement,denominationUsuelleEtablissement,dateDernierTraitementEtablissement,etablissementSiege,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,...,etatAdministratifEtablissement,activitePrincipaleEtablissement,caractereEmployeurEtablissement,codePaysEtrangerEtablissement,denominationUniteLegale,sexeUniteLegale,prenomUsuelUniteLegale,trancheEffectifsUniteLegale,categorieEntreprise,categorieJuridiqueUniteLegale
0,005520325,00552032500027,2018-12-20,NN,NaN,2025-12-06 04:31:28,True,36,AVENUE,JUNOT,...,A,70.10Z,N,NaN,ETABLISSEMENTS ADRIEN RIQUIER,NaN,NaN,12,PME,5710
1,005580501,00558050100046,1996-05-01,NN,NaN,2025-12-06 00:46:51,True,93,BOULEVARD,MALESHERBES,...,A,66.30Z,O,NaN,SGN PARIS INVESTISSEMENTS,NaN,NaN,NN,GE,5710
2,005720602,00572060200043,2007-03-19,NN,NaN,2025-12-05 20:27:36,True,14,RUE,MARC BLOCH,...,A,68.20B,N,NaN,SOC IMMOBILIERE MONOPRIX ABBEVILLE,NaN,NaN,NN,PME,5710
3,005720784,00572078400148,2011-01-01,NN,NaN,2025-12-06 04:56:10,False,1,IMPASSE,NICEPHORE NIEPCE,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710
4,005720784,00572078400163,2021-12-23,11,ETABLISSEMENTS DECAYEUX,2025-12-05 22:11:10,False,13,RUE,DES INDUSTRIES,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710


# Exclusion des entrepreneur individuels

In [6]:
df = df_fusion [df_fusion["categorieJuridiqueUniteLegale"] != 1000]

# Ajout variable nom entreprise

In [7]:
df["non_entreprise"] = np.where(
    df["denominationUsuelleEtablissement"].isna(),
    df["denominationUniteLegale"],
    df["denominationUsuelleEtablissement"]
)

# Ajout code activité niveau 1

In [8]:
df["niv1_activitePrincipaleEtablissement"] = df["activitePrincipaleEtablissement"].str[:2] 

# Ajout Lib codes activités

In [9]:
# Charger le fichier de correspondance
naf = pd.read_csv('data/int_courts_naf_rev_2 - NAF rév. 2.csv', sep=',', encoding='utf-8')

# Merge sur le code
df = df.merge(naf[["Code", "Intitule"]],
              left_on="niv1_activitePrincipaleEtablissement",
              right_on="Code",
              how="left")

# Renommer / nettoyer
df = df.rename(columns={"Intitule": "niv1_act_lib"}).drop(columns="Code")

In [10]:
# Charger le fichier de correspondance
naf = pd.read_csv('data/int_courts_naf_rev_2 - NAF rév. 2.csv', sep=',', encoding='utf-8')

# Merge sur le code
df = df.merge(naf[["Code", "Intitule"]],
              left_on="activitePrincipaleEtablissement",
              right_on="Code",
              how="left")

# Renommer / nettoyer
df = df.rename(columns={"Intitule": "act_lib"}).drop(columns="Code")

# Ajout département

In [11]:
df["departement"] = df["codePostalEtablissement"].str[:2] 

In [12]:
df["departement"].value_counts()

departement
75    815069
92    233651
93    182094
78    155005
94    154486
77    139449
95    131343
91    124269
Name: count, dtype: int64

# Ajout adresse

In [18]:
df['adresse'] = df['numeroVoieEtablissement'] +" "+ df['typeVoieEtablissement'] + " "  + df['libelleVoieEtablissement']

In [19]:
df.head()

,siren,siret,dateCreationEtablissement,trancheEffectifsEtablissement,dateDernierTraitementEtablissement,etablissementSiege,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codePostalEtablissement,...,prenomUsuelUniteLegale,trancheEffectifsUniteLegale,categorieEntreprise,categorieJuridiqueUniteLegale,non_entreprise,niv1_activitePrincipaleEtablissement,niv1_act_lib,act_lib,departement,adresse
0,005520325,00552032500027,2018-12-20,NN,2025-12-06 04:31:28,True,36,AVENUE,JUNOT,75018,...,NaN,12,PME,5710,ETABLISSEMENTS ADRIEN RIQUIER,70,Activités des sièges sociaux ; conseil de gestion,Activités des sièges sociaux,75,36 AVENUE JUNOT
1,005580501,00558050100046,1996-05-01,NN,2025-12-06 00:46:51,True,93,BOULEVARD,MALESHERBES,75008,...,NaN,NN,GE,5710,SGN PARIS INVESTISSEMENTS,66,Activités auxiliaires de services financiers e...,Gestion de fonds,75,93 BOULEVARD MALESHERBES
2,005720602,00572060200043,2007-03-19,NN,2025-12-05 20:27:36,True,14,RUE,MARC BLOCH,92110,...,NaN,NN,PME,5710,SOC IMMOBILIERE MONOPRIX ABBEVILLE,68,Activités immobilières,Location de terrains et d'autres biens immobil...,92,14 RUE MARC BLOCH
3,005720784,00572078400148,2011-01-01,NN,2025-12-06 04:56:10,False,1,IMPASSE,NICEPHORE NIEPCE,93290,...,NaN,31,PME,5710,ETABLISSEMENTS DECAYEUX,25,"Fabrication de produits métalliques, à l’excep...",Fabrication d'articles métalliques ménagers,93,1 IMPASSE NICEPHORE NIEPCE
4,005720784,00572078400163,2021-12-23,11,2025-12-05 22:11:10,False,13,RUE,DES INDUSTRIES,77460,...,NaN,31,PME,5710,ETABLISSEMENTS DECAYEUX,25,"Fabrication de produits métalliques, à l’excep...",Fabrication d'articles métalliques ménagers,77,13 RUE DES INDUSTRIES


# Supression colonne inutiles

In [ ]:
df = df.drop(columns=["denominationUsuelleEtablissement", "denominationUniteLegale",])

In [25]:
df.to_csv("data/resultat_idf.csv", index=False, encoding="utf-8")   

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1935366 entries, 0 to 1935365
Data columns (total 30 columns):
 #   Column                                  Dtype         
---  ------                                  -----         
 0   siren                                   str           
 1   siret                                   str           
 2   dateCreationEtablissement               object        
 3   trancheEffectifsEtablissement           str           
 4   dateDernierTraitementEtablissement      datetime64[us]
 5   etablissementSiege                      bool          
 6   numeroVoieEtablissement                 str           
 7   typeVoieEtablissement                   str           
 8   libelleVoieEtablissement                str           
 9   codePostalEtablissement                 str           
 10  libelleCommuneEtablissement             str           
 11  coordonneeLambertAbscisseEtablissement  float64       
 12  coordonneeLambertOrdonneeEtablissement  float64      

In [22]:
df["coordonneeLambertAbscisseEtablissement"].head()

0    651350.2009769329
1    649764.8893399736
2    649487.7305231433
3    668682.9795973853
4    680763.9703337984
Name: coordonneeLambertAbscisseEtablissement, dtype: str

In [23]:
# ── Conversion Lambert-93 → WGS84 ────────────────────────────────
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

col_x = 'coordonneeLambertAbscisseEtablissement'
col_y = 'coordonneeLambertOrdonneeEtablissement'

df[col_x] = pd.to_numeric(df[col_x], errors='coerce')
df[col_y] = pd.to_numeric(df[col_y], errors='coerce')

mask_gps = (
    df[col_x].notna() & df[col_y].notna() &
    df[col_x].between(100_000, 1_300_000) &
    df[col_y].between(6_000_000, 7_200_000)
)

lons, lats = transformer.transform(
    df.loc[mask_gps, col_x].values,
    df.loc[mask_gps, col_y].values
)
df['latitude']  = None
df['longitude'] = None
df.loc[mask_gps, 'latitude']  = lats
df.loc[mask_gps, 'longitude'] = lons

pct = mask_gps.sum() / len(df) * 100
print(f"📍 Coordonnées GPS : {mask_gps.sum():,} / {len(df):,} ({pct:.1f}%)")

📍 Coordonnées GPS : 1,851,640 / 1,935,366 (95.7%)


In [24]:
df["latitude"].head()

0    48.889129
1    48.878542
2    48.911559
3     48.96767
4    48.176379
Name: latitude, dtype: object